# Deep Hedging AI: Comparison to Black & Scholes (JAX port)

In this notebook we run Deep Hedging in a Black & Scholes world
_without statistical drift_. In this case we can compare hedging a vanilla call vs its perfect hedge.

### Proof that Deep Hedging should converge to the Black & Scholes price

We are using the notation from the mark down file https://github.com/hansbuehler/deephedging.
Let $Z_T$ be the claim in $T$. Assume that there are no trading cost and that the market is _complete_. This applies to Black & Scholes, and also to most other models typically used in quant finance.
In the limit of $dt\downarrow 0$ that menas there is a perfect replication strateg $\alpha^*$ such that
$$
    (*) \ \ \ \ Z_T \equiv z_0 + \sum_{t=0}^{T-1} \alpha_t^*(s_t)\ ( H_T - H_t)
$$
The constant $z_0$ is the risk-neutral price of $Z_T$.

Consider now our risk-adjusted return metric $U$ as used in Deep Hedging. We are trying to compute
$$
    \sup_a:\ U\left[\ Z_T + \sum_{t=0}^{T-1} a_t(s_t)\ ( H_T - H_t)\ \right]
$$
We now insert $(*)$ for $Z_T$. That gives
$$
    \sup_a:\ U\left[\ z_0 + \sum_{t=0}^{T-1} \alpha_t^*(s_t)\ ( H_T - H_t) + \sum_{t=0}^{T-1} a_t(s_t)\ ( H_T - H_t)\ \right]
$$
Because of _cash-invariance_ we can move $z_0$ out of $U$.
$$
(**) \ \ \ \    z_0 + \sup_a:\ U\left[\ \sum_{t=0}^{T-1} \left( \alpha_t^*(s_t) - a_t(s_t) \right)\ ( H_T - H_t) \ \right]
$$
In case that the underlying path generator is a risk-neutral measure we get $\mathrm{E}_t[ H_T - H_t ] = 0$ and therefore that $\mathrm{E}_t[ x_t(a_t) ( H_T - H_t)  ]=0$ for any possible strategy $x$.
Moreover, because $U$ has the property that $U(X) \leq E[X]$ we see that an optimal solutio for $(**)$ is achived in
$$
    a_t = -\alpha^*_t
$$
which means that the optimal solution to the Deep Hedging problem for _any_ monetary utility is the risk-neutral strategy.
We also note that the utility of $Z_T$ under the Deep Hedging is
$
z_0 + U(0) = z_0 
$
in other words it returns the risk-neutral price as well.

### Questions

* *Why are the hedges for vanilla options not limited by -1/+1:* the hedging signal from Deep Hedging is an interpolation. In the grapsh below we show both the hedge per one of the 10,000 path, and a binned interpolation. In order to increase precision, a bigger network needs to be trained. It is important to notice that the noise does not impede the quality of the hedge when measures via the utility $U$.
* *How does that compare to the results in the Deep Hedging paper:* in that paper we artificially restricted delta to -1/+1. This should have been noted in the model.

Port of `notebooks/trainer-bs.ipynb` by Hans Buehler, January 23 2023.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from deephedging.world import WorldConfig, simulate
from deephedging.model import AgentConfig, GymConfig
from deephedging.train import TrainConfig, fit
from deephedging.plotting import plot_training, plot_blackscholes

# DH_EPOCHS overrides every epoch count for quick smoke runs; DH_CHECKPOINTS="" turns checkpoints off.
epochs = lambda n: int(os.environ.get("DH_EPOCHS") or n)
CHECKPOINTS = os.environ.get("DH_CHECKPOINTS", "checkpoints") or None

In [ ]:
world_cfg = WorldConfig(samples=10000, steps=20, dt=5. / 250., drift=0., cost_s=0., black_scholes=True)
gym = GymConfig(agent=AgentConfig(depth=3, activation="softplus"), utility="cvar", lmbda=1.)
train_cfg = TrainConfig(epochs=epochs(20000), batch_size=2000)

# config_small_dt: the same setup with 100 daily steps
world_cfg_small_dt = WorldConfig(samples=10000, steps=20 * 5, dt=1. / 250., drift=0., cost_s=0., black_scholes=True)

## Deep Hedging in a Black & Scholes World - Big Steps

In [ ]:
world = simulate(world_cfg)
val_world = world.clone(samples=world.nSamples // 2, seed=1)
big = fit(gym, train_cfg, world, val_world, log_every=100, checkpoint_dir=CHECKPOINTS)
plot_training(big.result, big.train, big.val, world, val_world)
plt.show()

In [ ]:
plot_blackscholes(world, big.train, gym, vol=world_cfg.rvol)
plt.show()

## Deep Hedging in a Black & Scholes World - Small Steps (this blew up before 26000)

In [ ]:
world = simulate(world_cfg_small_dt)
val_world = world.clone(samples=world.nSamples // 2, seed=1)
small = fit(gym, train_cfg, world, val_world, log_every=100, checkpoint_dir=CHECKPOINTS)
plot_training(small.result, small.train, small.val, world, val_world)
plt.show()

In [ ]:
plot_blackscholes(world, small.train, gym, vol=world_cfg_small_dt.rvol)
plt.show()